# Prepare fingerprints — triplet-STDP extractor (VoxCeleb, one shard → one npz)

Runs the **exact current** SNN architecture of `training/tonotopic_plasticity_bound/train.py`
(N=128, 64 channels × 2 neurons/channel — sustained + onset, no phase; triplet STDP +
soft-refractory-gated lateral inhibition) as a frozen feature extractor, and writes one
fingerprint `.npz` per Kaggle input shard.

**Per-sample output** (see `_fingerprint_core.fingerprint_to_sample`):

| field | shape | dtype | meaning |
|---|---|---|---|
| `in_weights` | `(N,2,23,128)` | f16 | in→hid STDP weight type-images, silent cells zeroed |
| `hid_weights` | `(N,2,23,128)` | f16 | hid→hid STDP weight type-images, silent cells zeroed |
| `input_activity` | `(N,128)` | f16 | per-sample input firing rate ∈ [0,1] (final epoch only) |
| `hidden_activity` | `(N,128)` | f16 | per-sample hidden firing rate ∈ [0,1] (final epoch only) |
| `person_ids`, `session_ids`, `file_names`, `labels` | `(N,)` | str | speaker / session / wav filename / `person/session/wav` |

**Differences from `be_ann_w_fingerprints/prepare_fingerprints.ipynb`** (which ran an OLDER,
192-neuron/3-per-channel snapshot of the architecture):
- Matches the **current** `tonotopic_plasticity_bound/train.py`: N=128, no phase channel,
  current taus/gains/STDP amplitudes, `NORM_LIMIT_INH=0.40`, 25 ms normalisation period.
- Each wav is truncated to its first `CLIP_MS=2000` ms before encoding.
- All `N_EPOCHS=16` exposures run as **one continuous `net.run()` call** per wav — the input
  is tiled up front and network state is never reset between epochs (matches train.py's
  current persistent-state design), so there is no per-epoch rebuild/restore/reassign.
- The fingerprint is the network's state after that single run, i.e. the **last epoch's
  result only** — no snapshot collection, no averaging across epochs.
- **New**: `hid_weights` (hidden→hidden STDP weights) is now part of the fingerprint,
  transformed into the same `(type, channel-offset, N_H)` layout as `in_weights`.
- A short dummy `net.run()` warms the Cython codegen cache once per worker (and once in the
  parent process before workers spawn), so every real per-wav run reuses compiled code
  instead of triggering compilation.

**Workflow** — you have many shards and run this notebook once per shard:
1. Edit the **config cell**: point `INPUT_ROOT` at the shard folder, set `OUTPUT_NAME`
   (e.g. `vox2_200person_shard01_fingerprints.npz`, or `vox1_...` for the test set — for
   VoxCeleb1 shards, point `INPUT_ROOT` at the specific `dev_NN` folder since that corpus has
   an extra directory level).
2. Run all cells. The npz lands in `/kaggle/working/`; download it and upload the whole
   collection as one Kaggle dataset for the downstream ECAPA/FiLM notebook.

The `%%writefile` cells recreate the extractor + audio encoder as real modules in the
working dir so that multiprocessing `spawn` workers can import them (a notebook has no
importable `__main__`). Requires `gammatone` (pip below) and `ffmpeg` (preinstalled on
Kaggle) to decode `.m4a`.


In [ ]:
# ── Setup: gammatone filterbank (Kaggle already ships ffmpeg, librosa, soundfile, brian2)
!pip -q install gammatone 2>/dev/null || pip -q install git+https://github.com/detly/gammatone.git
!pip -q install brian2
import shutil
assert shutil.which("ffmpeg"), "ffmpeg not on PATH — needed to decode .m4a"
print("ffmpeg:", shutil.which("ffmpeg"))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG  —  the only cell you edit between shards
# ══════════════════════════════════════════════════════════════════════════════
import os

# Kaggle input shard folder: {INPUT_ROOT}/{person_id}/{session_id}/{wav}.m4a
INPUT_ROOT  = "/kaggle/input/datasets/qphulong/vox2-voices-200person-shard01"

# Output npz name (kept as-is on /kaggle/working). Use a per-shard name so many
# shards can be uploaded together later:
#   dev  set : vox2_200person_shard01_fingerprints.npz, ..._shard02_..., ...
#   test set : vox1_200person_shard01_fingerprints.npz, ...
# VoxCeleb1 shards have an extra dev_NN level — point INPUT_ROOT at the specific
# dev_NN folder (e.g. .../vox1-voices/dev_01) so the {person}/{session}/{wav}
# scan below still applies unchanged.
OUTPUT_NAME = "vox2_200person_shard01_fingerprints.npz"

WORK_DIR   = "/kaggle/working"
WORKERS    = os.cpu_count() or 2   # parallel Brian2 worker processes (Kaggle ~4)
AUDIO_EXTS = (".m4a", ".wav")      # VoxCeleb1+2 = m4a; repo-local test wavs may be .wav

# One utterance per session is picked at random (seeded) — same SEED and same
# picking rule as training/ecapa_film_snn/ecapa_baseline.ipynb's own corpus-wide
# protocol, so both pipelines independently agree on "1 random utterance per
# session" without needing to see each other's output. The exact (person_id,
# session_id, file_name) actually picked is what gets recorded as metadata below,
# so downstream code never needs to reproduce this pick itself — it just reads it.
SEED = 1234

print(f"INPUT_ROOT  = {INPUT_ROOT}")
print(f"OUTPUT_NAME = {OUTPUT_NAME}")
print(f"SEED        = {SEED}  |  WORKERS = {WORKERS}")

### Write the extractor modules to the working dir (imported by spawn workers)

In [ ]:
%%writefile audio_utils.py
import shutil
import subprocess

import numpy as np
import librosa
import soundfile as sf
from gammatone.filters import centre_freqs, make_erb_filters, erb_filterbank


def _ffprobe_sample_rate(path):
    """Native sample rate of `path` via ffprobe, or None if it can't be determined."""
    ffprobe = shutil.which("ffprobe")
    if ffprobe is None:
        return None
    try:
        out = subprocess.run(
            [ffprobe, "-v", "error", "-select_streams", "a:0",
             "-show_entries", "stream=sample_rate",
             "-of", "default=noprint_wrappers=1:nokey=1", path],
            check=True, capture_output=True, text=True,
        ).stdout.strip().splitlines()
        return int(out[0]) if out else None
    except (subprocess.CalledProcessError, ValueError):
        return None


def _ffmpeg_decode(path, sr):
    """Decode any ffmpeg-readable container (m4a/aac/mp3/...) to a mono float32
    waveform. If `sr` is None the native rate is probed and preserved; otherwise
    ffmpeg's resampler outputs directly at `sr`."""
    ffmpeg = shutil.which("ffmpeg")
    if ffmpeg is None:
        raise RuntimeError(
            f"Cannot decode {path!r}: libsndfile failed and ffmpeg is not on PATH. "
            "Install ffmpeg to read m4a/aac audio."
        )
    target_sr = sr if sr is not None else (_ffprobe_sample_rate(path) or 16000)
    proc = subprocess.run(
        [ffmpeg, "-nostdin", "-loglevel", "error", "-i", path,
         "-ac", "1", "-ar", str(target_sr),
         "-f", "f32le", "-acodec", "pcm_f32le", "-"],
        capture_output=True,
    )
    if proc.returncode != 0:
        raise RuntimeError(
            f"ffmpeg failed to decode {path!r}: "
            f"{proc.stderr.decode('utf-8', 'ignore').strip()}"
        )
    y = np.frombuffer(proc.stdout, dtype="<f4").astype(np.float32)
    return y, target_sr


def load_audio(path, sr=16000):
    """Load `path` to a mono float32 waveform at `sr` Hz (native rate if `sr` is None).

    Format-robust replacement for ``librosa.load``: wav/flac/ogg are decoded by
    libsndfile (soundfile); m4a/aac and any container libsndfile can't open are
    decoded via ffmpeg. This avoids librosa's audioread m4a fallback, which is
    deprecated and slated for removal in librosa 1.0 (and emits a warning per file).
    """
    try:
        y, sr_native = sf.read(path, dtype="float32", always_2d=False)
    except Exception:
        return _ffmpeg_decode(path, sr)
    if y.ndim > 1:                       # multi-channel -> mono (match librosa default)
        y = y.mean(axis=1)
    if sr is not None and sr_native != sr:
        y = librosa.resample(y, orig_sr=sr_native, target_sr=sr)
    return np.ascontiguousarray(y, dtype=np.float32), (sr if sr is not None else sr_native)


def load_mel_spectrogram(
    wav_path: str,
    n_mels: int = 256,
    fmax: int | None = 8000,
    target_frames_per_second: int = 1000,
    normalize: bool = True,
):
    audio, sr = load_audio(wav_path, sr=None)

    hop_length = int(sr / target_frames_per_second)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=n_mels,
        fmax=fmax,
        hop_length=hop_length
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)

    if normalize:
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)

    return mel_db, sr

def auditory_frontend(
    audio_path,
    sr=16000,
    num_filters=100,
    f_min=50,
    alpha=1.0,
    norm_percentile=99.0,
    clip_val=1.0,
    per_channel=False,
    eps=1e-6,
):
    """
    Encode an audio waveform into auditory-inspired spike features

    This function implements a biologically inspired auditory pipeline:
    waveform → gammatone filterbank → upstream percentile normalization →
    inner hair cell compression → onset detection → phase signal.

    Normalization is applied **once, upstream** to the (signed) filterbank output,
    before any nonlinearity. Because `log1p(alpha * x)` is not scale-invariant, the
    signal must be brought to a known scale *before* the log so compression is
    consistent across utterances. E, dE and phase are then all derived from the same
    normalized signal — their relative balance is therefore set only by the downstream
    gains, not by independent per-feature normalizations.

    Parameters
    ----------
    audio_path : str
        Path to the input audio file.

    sr : int, default=16000
        Target sampling rate for loading audio.

    num_filters : int, default=100
        Number of ERB-spaced gammatone filters (frequency channels).

    f_min : float, default=50
        Minimum center frequency (Hz) of the filterbank.

    alpha : float, default=1.0
        Compression strength for inner hair cell log compression:
        E = log1p(alpha * |signal_norm|).

    norm_percentile : float, default=99.0
        Percentile of |filterbank output| used as the normalization scale. Robust to
        the loudest transients (top 1% at 99) compared to a plain max.

    clip_val : float, default=1.0
        After dividing by the percentile scale, the normalized signal is clipped to
        [-clip_val, clip_val]. This bounds the input to the log and saturates the
        loudest excursions.

    per_channel : bool, default=False
        If False (default), one global percentile scalar is computed over the whole
        (n_channels, T) magnitude array — this preserves cross-channel relative energy
        (formant/timbre structure useful for speaker discrimination). If True, the
        percentile is computed per channel, equalizing quiet and loud bands.

    eps : float, default=1e-6
        Small constant to avoid division by zero.

    Returns
    -------
    dict
        Dictionary containing encoded auditory representations:

        - "E" : np.ndarray (n_channels, T)
            Log-compressed cochlear energy (IHC output), full-wave rectified.

        - "dE" : np.ndarray (n_channels, T)
            Onset detection signal (half-wave rectified temporal derivative of E).

        - "phase" : np.ndarray (n_channels, T)
            Negative half-wave of the normalized filterbank output. Complementary in
            polarity to E's full-wave energy, so it is not redundant with E.

        - "cf" : np.ndarray (n_channels,)
            Center frequencies of filterbank channels (low → high)

        - "sr" : int
            Sampling rate of processed audio

    Notes
    -----
    Processing pipeline:

    1. Audio loading
    2. ERB-spaced gammatone filterbank
    3. Upstream percentile normalization + clip (on the signed signal)
    4. Inner hair cell log compression (full-wave): E = log1p(alpha * |sig_norm|)
    5. Onset detection via positive temporal derivative of E
    6. Phase signal: negative half-wave of sig_norm

    All channel outputs are ordered from **low → high frequency**.
    """

    # ==============================
    # 1. Load audio
    # ==============================
    signal, sr = load_audio(audio_path, sr=sr)

    # ==============================
    # 2. Gammatone filterbank
    # ==============================
    cf = centre_freqs(sr, num_filters, f_min)
    erb_filters = make_erb_filters(sr, cf)

    filtered_signals = erb_filterbank(signal, erb_filters)

    # reorder HIGH→LOW → LOW→HIGH
    cf = cf[::-1]
    filtered_signals = filtered_signals[::-1]

    signals = filtered_signals
    n_channels, T = signals.shape

    # ==============================
    # 3. Upstream percentile normalization (on the signed signal, before any
    #    nonlinearity). One scale derived from |signals|, then clip. This keeps
    #    the log compression in a consistent regime across utterances and puts
    #    E / dE / phase on a single shared reference frame.
    # ==============================
    if per_channel:
        scale = np.percentile(np.abs(signals), norm_percentile, axis=1, keepdims=True)
    else:
        scale = np.percentile(np.abs(signals), norm_percentile)
    sig_n = np.clip(signals / (scale + eps), -clip_val, clip_val)

    # ==============================
    # 4. Inner Hair Cell Compression (full-wave)
    # ==============================
    E = np.log1p(alpha * np.abs(sig_n))

    # ==============================
    # 5. Onset detection (positive temporal derivative of E)
    # ==============================
    dE = np.diff(E, axis=1, prepend=E[:, :1])
    dE[dE < 0] = 0

    # ==============================
    # 6. Phase signal: negative half-wave of the normalized signal.
    #    Complementary in polarity to E's full-wave energy → not redundant with E.
    # ==============================
    phase_signal = np.maximum(-sig_n, 0)

    return {
        "E": E,
        "dE": dE,
        "phase": phase_signal,
        "cf": cf,
        "sr": sr,
    }


In [ ]:
%%writefile spike_encoding.py
import numpy as np
from audio_utils import auditory_frontend

def compute_spike_input_current(
    audio_path,
    sustained_per_band=5,
    onset_per_band=2,
    phase_per_band=2,
    scale=1,
    sust_gain=1.0,
    onset_gain=2.0,
    phase_gain=1.0,
    sust_spread_min=0.6,
    sust_spread_max=1.4,
    audio_sample_rate=16000,
    simulation_sample_rate=1000,
    num_filters=100,
    norm_percentile=99.0,
    clip_val=1.0,
    per_channel=False,
):
    """
    Convert an audio file into a downsampled input current matrix for a spiking neural network

    This function takes auditory features produced by `auditory_frontend()` and expands
    them into multiple neuron types per cochlear frequency band. Each neuron type
    represents different auditory response characteristics inspired by biological
    auditory nerve fibers.

    Pipeline
    --------
    1. Audio → auditory feature extraction via `auditory_frontend()`
    2. Obtain three feature maps:
        - E     : sustained energy (IHC compressed output)
        - dE    : onset energy (positive temporal derivative)
        - phase : rectified gammatone signal
    3. Generate multiple neurons per frequency band:
        - sustained neurons (energy response, spread across gain multipliers)
        - onset neurons (transient response)
        - phase neurons (phase locking)
    4. Apply gain scaling and small Gaussian noise.
    5. Downsample from `audio_sample_rate` to `simulation_sample_rate` by
       block-averaging across the decimation factor.
    6. Return a time-varying current matrix suitable for driving LIF neurons.

    Parameters
    ----------
    audio_path : str
        Path to the input audio (.wav) file.

    sustained_per_band : int, default=5
        Number of neurons per frequency band that encode sustained energy (E).

    onset_per_band : int, default=2
        Number of neurons per band that encode onset activity (dE).

    phase_per_band : int, default=2
        Number of neurons per band that encode phase-locking signals.

    scale : float, default=1
        Global gain multiplier applied to all input currents.

    sust_gain : float, default=1.3
        Gain factor for sustained-energy neurons.

    onset_gain : float, default=2.0
        Gain factor for onset neurons.

    phase_gain : float, default=1.0
        Gain factor for phase-locking neurons.

    sust_spread_min : float, default=0.6
        Minimum multiplicative factor applied across sustained neurons within a band.

    sust_spread_max : float, default=1.4
        Maximum multiplicative factor applied across sustained neurons within a band.

    audio_sample_rate : int, default=16000
        Sampling rate (Hz) of the raw audio and the auditory feature maps produced
        by `auditory_frontend()`.

    simulation_sample_rate : int, default=1000
        Target sampling rate (Hz) for the Brian2 simulation (i.e. 1 / defaultclock.dt).
        The current matrix is downsampled from `audio_sample_rate` to this rate by
        block-averaging. Must evenly divide `audio_sample_rate`.

    norm_percentile : float, default=99.0
        Percentile used by `auditory_frontend` to normalize the filterbank output
        before the log nonlinearity.

    clip_val : float, default=1.0
        Clip bound applied to the normalized filterbank signal in `auditory_frontend`.

    per_channel : bool, default=False
        If True, `auditory_frontend` normalizes per channel instead of globally.

    Returns
    -------
    I_sim : np.ndarray, shape (N_in, T_sim)
        Simulation-ready input current matrix, where
        T_sim = T // decimation_factor.

    T_sim : int
        Number of time steps after downsampling, corresponding to the
        total simulation duration in Brian2 timesteps.
    """

    feats = auditory_frontend(
        audio_path,
        num_filters=num_filters,
        norm_percentile=norm_percentile,
        clip_val=clip_val,
        per_channel=per_channel,
    )

    E = feats["E"]
    dE = feats["dE"]
    phase = feats["phase"]

    n_channels, T = E.shape

    g_sust = sust_gain
    g_onset = onset_gain
    g_phase = phase_gain

    neurons_per_band = sustained_per_band + onset_per_band + phase_per_band
    N_in = n_channels * neurons_per_band

    I = np.zeros((N_in, T), dtype=np.float32)

    idx = 0
    for ch in range(n_channels):

        spread = np.linspace(sust_spread_min, sust_spread_max, sustained_per_band)
        for mult in spread:
            I[idx] = g_sust * mult * scale * E[ch]
            idx += 1

        for _ in range(onset_per_band):
            I[idx] = g_onset * scale * dE[ch]
            idx += 1

        for _ in range(phase_per_band):
            I[idx] = g_phase * scale * phase[ch]
            idx += 1

    I += 0.01 * np.random.randn(*I.shape).astype(np.float32)
    assert audio_sample_rate % simulation_sample_rate == 0, (
        f"audio_sample_rate ({audio_sample_rate}) must be divisible by "
        f"simulation_sample_rate ({simulation_sample_rate})"
    )
    decimation_factor = audio_sample_rate // simulation_sample_rate
    # Trim to nearest multiple so reshape never fails
    T_trim = (T // decimation_factor) * decimation_factor
    I_sim  = I[:, :T_trim].reshape(N_in, -1, decimation_factor).mean(axis=2)
    T_sim  = I_sim.shape[1]

    return I_sim, T_sim


In [ ]:
%%writefile _fingerprint_core.py
"""
_fingerprint_core.py
=====================
Fingerprint extractor core for ecapa_film_snn.

The Brian2 network is IDENTICAL to the CURRENT training/tonotopic_plasticity_bound/train.py
(as of 2026-08-31): N=128 (64 channels x 2 neurons/channel: sustained + onset, NO phase
channel), adaptive-LIF input, adaptive-threshold LIF hidden (membrane noise + soft-refractory
trace_r replacing hard refractory), excitatory input->hidden triplet STDP (Pfister-Gerstner
slow detectors r2/o2, A3pre/A3post on top of the pair rule) tonotopically bounded by
max(0, 1-(d_ch/R)^p), inhibitory hidden->hidden STDP (Jaccard-scaled connectivity, current
gated by the postsynaptic soft-refractory trace), homeostatic L1 column normalisation every
25ms (exc cap 1.0455, inh cap 0.40).

Differences from be_ann_w_fingerprints/_fingerprint_core_2.py (which snapshotted an OLDER,
192-neuron/3-per-channel version of tonotopic_plasticity_bound):
  - N_IN = N_H = 128, N_PER_CHANNEL = 2 (sustained, onset — no phase neurons)
  - CLIP_MS = 2000: only the first 2s of each wav is used
  - N_EPOCHS = 16, run as ONE continuous net.run() per wav (input tiled N_EPOCHS times up
    front, network state never reset between epochs — matches train.py's current design).
    No per-epoch Python loop, no per-epoch restore/reassign.
  - The fingerprint is the network's state after that single run — i.e. the LAST epoch's
    result. No snapshot collection, no averaging.
  - Fingerprint now has FOUR components per sample: in->hid weights, hid->hid weights (new),
    input firing rate, hidden firing rate — see fingerprint_to_sample().
  - A short dummy net.run() warms the Cython codegen cache once per worker (and once in the
    parent, before workers spawn) so the real per-wav runs skip recompilation.

NUM_EPOCHS/CLIP_MS are baked-in literals, not imported from train.py (a Kaggle notebook can't
reach the repo). If you retune tonotopic_plasticity_bound/train.py, re-sync this file by hand.

Worker helpers (_init_worker / process_one) live here too so multiprocessing `spawn` workers
can import them from a real module (a notebook has no importable __main__).

Import this module BEFORE numpy in driver code so the BLAS-thread pinning below takes
effect (prevents oversubscription when many worker processes run in parallel).
"""

import os

# ── Pin BLAS threads BEFORE numpy is imported anywhere in the process ──────────
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")

import sys
from types import SimpleNamespace

import numpy as np

# Make the sibling encoder modules importable whether imported by path or re-imported
# by a spawned worker (cwd is not guaranteed to be on sys.path in a spawn child).
_HERE = os.path.dirname(os.path.abspath(__file__))
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)
from spike_encoding import compute_spike_input_current

from brian2 import (
    NeuronGroup, Synapses, SpikeMonitor, TimedArray, Network, network_operation,
    defaultclock, prefs, BrianLogger, ms, second,
)

# ── Brian2 codegen / logging prefs ─────────────────────────────────────────────
prefs.codegen.target = 'cython'                          # JIT to C (gcc present)
prefs.codegen.runtime.cython.multiprocess_safe = True    # safe parallel build cache
BrianLogger.suppress_name('method_choice')
BrianLogger.suppress_name('unused_brian_object')         # warm-compile net is discarded
prefs.logging.console_log_level = 'ERROR'                # quiet across many workers

# ═══════════════════════════════════════════════════════════════════════════════
# Hyperparameters — verbatim from tonotopic_plasticity_bound/train.py as of 2026-08-31
# ═══════════════════════════════════════════════════════════════════════════════

N_IN = 128   # 64 channels x 2 neurons/channel (sustained + onset, no phase)
N_H  = 128

DT_SIM = 1 * ms

# -- Training exposure (fingerprint-pipeline specific; not part of train.py's own state) --
CLIP_MS  = 2000   # only the first CLIP_MS of each wav is used
N_EPOCHS = 16     # full-(truncated-)clip exposures, run as ONE continuous net.run()

# -- Input layer (adaptive LIF) --
tau_m       = 40 * ms
tau_a       = 40 * ms
beta        = 3.5
tau_current = 1 * ms
v_th_in     = 1.0

# -- Hidden layer (adaptive-threshold LIF) --
tau_h    = 150 * ms
tau_vth  = 60 * ms
vth_rest = 0.6
vth_init = 0.6
vth_jump = 1.0

# -- Soft refractory (hidden only; replaces hard refractory) --
tau_r = 10 * ms

# -- Membrane noise (hidden only) --
sigma_noise = 0.03 * second**(-0.5)

# -- STDP (excitatory pair) --
taupre  = 20 * ms
taupost = 20 * ms

# -- Triplet STDP (excitatory; Pfister-Gerstner, on top of the pair rule) --
tau_x         = 100 * ms   # slow presynaptic detector r2
tau_y         = 125 * ms   # slow postsynaptic detector o2
A3PRE_CENTER  =  0.004     # triplet LTP amplitude (pre-post-post), > 0
A3POST_CENTER = -0.002     # triplet LTD amplitude (post-pre-pre), < 0

# -- Excitatory weight bounds --
wmin = 0.0

# -- Excitatory synapse --
WMAX_CENTER  = 1.0
APRE_CENTER  =  0.008
APOST_CENTER = -0.0096

# -- Inhibitory lateral synapse --
W_INH_CENTER = 1.0
W_INH_MIN    = 0.0
APRE_INH     = 0.004
APOST_INH    = -0.0048

# -- Channel layout --
N_CHANNELS    = 64
N_PER_CHANNEL = N_IN // N_CHANNELS   # 2

# -- Tonotopic plasticity (polynomial decay: max(0, 1-(d_channel/R)^p)) --
R_EXC_CHANNEL = 11
p_EXC         = 3

# -- Homeostatic normalisation --
NORM_LIMIT_EXC = 1.0455
NORM_LIMIT_INH = 0.40
NORM_DT        = 25 * ms

# -- Audio encoder kwargs (verbatim from train.py's compute_spike_input_current call) --
ENCODER_KWARGS = dict(
    scale=1.0,
    num_filters=64,
    sustained_per_band=1,
    onset_per_band=1,
    phase_per_band=0,
    sust_gain=0.3,
    onset_gain=3.0,
    sust_spread_min=1,
    sust_spread_max=1,
)

# -- Tensor transform layout --
OFFSETS = np.arange(-R_EXC_CHANNEL, R_EXC_CHANNEL + 1)   # -11..+11 (23 values)
N_OFF   = len(OFFSETS)

# Per-sample output shapes (one fingerprint):
WEIGHTS_SHAPE  = (N_PER_CHANNEL, N_OFF, N_H)   # (2, 23, 128) — both in->hid and hid->hid
ACTIVITY_SHAPE = (N_IN,)                        # (128,)

# ═══════════════════════════════════════════════════════════════════════════════
# Read-only module-level precompute (deterministic, fork-safe)
# ═══════════════════════════════════════════════════════════════════════════════

# -- Tonotopic excitatory matrices (input -> hidden) -------------------------------
_ch_i    = (np.arange(N_IN) // N_PER_CHANNEL).reshape(-1, 1)
_ch_j    = (np.arange(N_H)  // N_PER_CHANNEL).reshape(1, -1)
_dist_ch = np.abs(_ch_i - _ch_j)
_dist_ch = np.minimum(_dist_ch, N_CHANNELS - _dist_ch)            # circular
_topo_exc = np.maximum(0.0, 1.0 - (_dist_ch / R_EXC_CHANNEL) ** p_EXC)
_mask_ih  = _dist_ch <= R_EXC_CHANNEL
_SRC_IH, _TGT_IH = np.where(_mask_ih)

_WMAX_MATRIX   = WMAX_CENTER   * _topo_exc
_APRE_MATRIX   = APRE_CENTER   * _topo_exc
_APOST_MATRIX  = APOST_CENTER  * _topo_exc
_A3PRE_MATRIX  = A3PRE_CENTER  * _topo_exc   # triplet LTP amplitude (topo-scaled)
_A3POST_MATRIX = A3POST_CENTER * _topo_exc   # triplet LTD amplitude (topo-scaled)
del _ch_i, _ch_j, _dist_ch, _topo_exc

# -- Jaccard inhibitory matrices (hidden -> hidden) --------------------------------
_ch_h       = np.arange(N_H) // N_PER_CHANNEL
_dist_ch_hh = np.abs(_ch_h.reshape(-1, 1) - _ch_h.reshape(1, -1))
_dist_ch_hh = np.minimum(_dist_ch_hh, N_CHANNELS - _dist_ch_hh)   # circular
_window_size = 2 * R_EXC_CHANNEL + 1
_overlap_ch  = np.maximum(0, _window_size - _dist_ch_hh)
_jaccard     = np.where(_overlap_ch > 0,
                        _overlap_ch / (_window_size + _dist_ch_hh), 0.0)
_mask_hh  = (_overlap_ch > 0) & (~np.eye(N_H, dtype=bool))
_SRC_HH, _TGT_HH = np.where(_mask_hh)

_WMAX_INH_MATRIX  = W_INH_CENTER * _jaccard
_APRE_INH_MATRIX  = APRE_INH     * _jaccard
_APOST_INH_MATRIX = APOST_INH    * _jaccard
del _ch_h, _dist_ch_hh, _overlap_ch, _jaccard

# -- Shared initial weight matrices (per-wav starting point, verbatim train.py init) --
# Excitatory: formula-shaped, column-normalised to NORM_LIMIT_EXC.
W_IH_INIT = np.zeros((N_IN, N_H))
W_IH_INIT[_SRC_IH, _TGT_IH] = _WMAX_MATRIX[_SRC_IH, _TGT_IH]
for _j in range(N_H):
    _rows = _SRC_IH[_TGT_IH == _j]
    _wsum = W_IH_INIT[_rows, _j].sum()
    if _wsum > 0:
        W_IH_INIT[_rows, _j] *= NORM_LIMIT_EXC / _wsum

# Inhibitory: uniform random init in [0.01, 0.02] on connected positions, drawn once
# from a fixed seed so every wav / worker starts from the same state.
_rng = np.random.RandomState(42)
W_HH_INIT = np.zeros((N_H, N_H))
W_HH_INIT[_SRC_HH, _TGT_HH] = _rng.uniform(0.01, 0.02, size=_SRC_HH.shape[0])

# -- Transform index maps ------------------------------------------------------------
# IN_IDX[t, o, j]: input-neuron index at type t, channel-offset o from hidden neuron j's
# channel. HH_IDX[t, o, j]: same formula, indexing into the (N_H, N_H) hid->hid matrix —
# both source layers share the same N_PER_CHANNEL/channel layout, so one index map serves
# both.
_ch_j_row = (np.arange(N_H) // N_PER_CHANNEL)                       # (N_H,)
_off_ch   = (_ch_j_row[None, :] + OFFSETS[:, None]) % N_CHANNELS    # (23, N_H)
IN_IDX = np.stack([_off_ch * N_PER_CHANNEL + t for t in range(N_PER_CHANNEL)])  # (2,23,N_H)
HH_IDX = IN_IDX
_J_ROW = np.broadcast_to(np.arange(N_H), WEIGHTS_SHAPE)             # (2,23,N_H)
del _ch_j_row, _off_ch


# ═══════════════════════════════════════════════════════════════════════════════
# Network construction  (call once per process)
# ═══════════════════════════════════════════════════════════════════════════════

def build_network():
    """Build the Brian2 network fresh and return a handle namespace.

    Must be called once per worker process (Brian2 objects are not safe to fork
    already-built). The returned handle is passed to train_fingerprint().
    """
    defaultclock.dt = DT_SIM

    # ── Input neurons ──────────────────────────────────────────────────────────
    eqs_in = """
    dv/dt = (-v - a) / tau_m + I_timed(t, i) / tau_current : 1
    da/dt = -a / tau_a : 1
    """
    G_in = NeuronGroup(N_IN, eqs_in, threshold="v > v_th_in",
                       reset="v=0; a+=beta", refractory=2 * ms, method="euler")
    G_in.namespace["I_timed"] = TimedArray(np.zeros((1, N_IN), dtype=float), dt=DT_SIM)

    # ── Hidden neurons ─────────────────────────────────────────────────────────
    eqs_h = f"""
    dv/dt       = -v / tau_h + sigma_noise * xi                       : 1
    dvth/dt     = -(vth - {vth_rest}) / tau_vth                       : 1
    dtrace_r/dt = -trace_r / tau_r                                    : 1
    """
    G_h = NeuronGroup(N_H, eqs_h, threshold="v > vth",
                      reset=f"v=0; vth=vth+{vth_jump}; trace_r=1;", method="euler")

    # ── Excitatory TRIPLET STDP synapses: input → hidden ───────────────────────
    stdp_model = """
    w          : 1
    dapre/dt   = -apre  / taupre  : 1 (event-driven)
    dapost/dt  = -apost / taupost : 1 (event-driven)
    dr1/dt     = -r1 / taupre      : 1 (event-driven)
    dr2/dt     = -r2 / tau_x       : 1 (event-driven)
    do1/dt     = -o1 / taupost     : 1 (event-driven)
    do2/dt     = -o2 / tau_y       : 1 (event-driven)
    wmax_syn   : 1
    Apre_syn   : 1
    Apost_syn  : 1
    A3pre_syn  : 1
    A3post_syn : 1
    """
    on_pre  = (f"v_post += w * (1 - trace_r_post)\n"
               f"apre += Apre_syn\n"
               f"w = clip(w + apost*(w-{wmin}) + A3post_syn*o1*r2*(w-{wmin}), {wmin}, wmax_syn)\n"
               f"r1 += 1\n"
               f"r2 += 1")
    on_post = (f"apost += Apost_syn\n"
               f"w = clip(w + apre*(wmax_syn-w) + A3pre_syn*r1*o2*(wmax_syn-w), {wmin}, wmax_syn)\n"
               f"o1 += 1\n"
               f"o2 += 1")

    S_ih = Synapses(G_in, G_h, model=stdp_model, on_pre=on_pre, on_post=on_post)
    S_ih.connect(i=_SRC_IH, j=_TGT_IH)
    src_ih = np.array(S_ih.i)
    tgt_ih = np.array(S_ih.j)
    S_ih.wmax_syn   = _WMAX_MATRIX[src_ih, tgt_ih]
    S_ih.Apre_syn   = _APRE_MATRIX[src_ih, tgt_ih]
    S_ih.Apost_syn  = _APOST_MATRIX[src_ih, tgt_ih]
    S_ih.A3pre_syn  = _A3PRE_MATRIX[src_ih, tgt_ih]
    S_ih.A3post_syn = _A3POST_MATRIX[src_ih, tgt_ih]

    # ── Inhibitory lateral STDP synapses: hidden → hidden (soft-refractory gated) ─
    stdp_inh_model = """
    w_inh          : 1
    dapre_inh/dt   = -apre_inh  / taupre  : 1 (event-driven)
    dapost_inh/dt  = -apost_inh / taupost : 1 (event-driven)
    wmax_inh_syn   : 1
    Apre_inh_syn   : 1
    Apost_inh_syn  : 1
    """
    on_pre_inh  = (f"v_post -= w_inh * (1 - trace_r_post)\n"
                   f"apre_inh += Apre_inh_syn\n"
                   f"w_inh = clip(w_inh + apost_inh*(w_inh-{W_INH_MIN}), {W_INH_MIN}, wmax_inh_syn)")
    on_post_inh = (f"apost_inh += Apost_inh_syn\n"
                   f"w_inh = clip(w_inh + apre_inh*(wmax_inh_syn-w_inh), {W_INH_MIN}, wmax_inh_syn)")

    S_hh = Synapses(G_h, G_h, model=stdp_inh_model, on_pre=on_pre_inh, on_post=on_post_inh)
    S_hh.connect(i=_SRC_HH, j=_TGT_HH)
    src_hh = np.array(S_hh.i)
    tgt_hh = np.array(S_hh.j)
    S_hh.wmax_inh_syn  = _WMAX_INH_MATRIX[src_hh, tgt_hh]
    S_hh.Apre_inh_syn  = _APRE_INH_MATRIX[src_hh, tgt_hh]
    S_hh.Apost_inh_syn = _APOST_INH_MATRIX[src_hh, tgt_hh]
    S_hh.w_inh         = W_HH_INIT[src_hh, tgt_hh]

    # ── Spike monitors (record=True: we need actual spike TIMES, not just totals,
    #    to isolate the final epoch's window after one continuous multi-epoch run) ─
    spike_in  = SpikeMonitor(G_in)
    spike_hid = SpikeMonitor(G_h)

    # ── Vectorised L1 normalisation every 25 ms ─────────────────────────────────
    wmax_syn_arr     = np.array(S_ih.wmax_syn)
    wmax_inh_syn_arr = np.array(S_hh.wmax_inh_syn)

    @network_operation(dt=NORM_DT, when='end')
    def normalize_weights():
        w = np.array(S_ih.w)
        col_sum = np.bincount(tgt_ih, weights=w, minlength=N_H)
        scale = np.where(col_sum > NORM_LIMIT_EXC, NORM_LIMIT_EXC / col_sum, 1.0)
        S_ih.w[:] = np.clip(w * scale[tgt_ih], wmin, wmax_syn_arr)

        wi = np.array(S_hh.w_inh)
        col_sum_i = np.bincount(tgt_hh, weights=wi, minlength=N_H)
        scale_i = np.where(col_sum_i > NORM_LIMIT_INH, NORM_LIMIT_INH / col_sum_i, 1.0)
        S_hh.w_inh[:] = np.clip(wi * scale_i[tgt_hh], W_INH_MIN, wmax_inh_syn_arr)

    net = Network(G_in, G_h, S_ih, S_hh, spike_in, spike_hid, normalize_weights)
    G_h.vth = vth_init
    net.store('init')   # clean snapshot: clock=0, v=0, a=0, vth=vth_init, monitors empty

    return SimpleNamespace(
        net=net, G_in=G_in, G_h=G_h, S_ih=S_ih, S_hh=S_hh,
        src_ih=src_ih, tgt_ih=tgt_ih, src_hh=src_hh, tgt_hh=tgt_hh,
        spike_in=spike_in, spike_hid=spike_hid,
    )


def warmup(h):
    """Force Brian2's Cython codegen to compile NOW, via a short dummy run, instead
    of lazily on the first real wav — then restore to a clean state.

    Compiled code is cached to disk (multiprocess_safe=True), so calling this once in
    the parent process (before spawning workers) and once per worker at startup means
    only the very first call actually pays the compile cost; every real per-wav run
    after that reuses the cached extension — "first train to warm up" so the real runs
    are fast.

    Weights are seeded with the real W_IH_INIT/W_HH_INIT (not left at Brian2's default
    zero-init) before running — otherwise normalize_weights()'s column sums are all
    zero during this dummy run and every NORM_LIMIT_EXC/col_sum division warns
    (RuntimeWarning: divide by zero). Harmless either way (train_fingerprint() always
    re-seeds weights before its own real run), but seeding here keeps the warm-up run
    numerically representative and silent.
    """
    h.S_ih.w     = W_IH_INIT[h.src_ih, h.tgt_ih]
    h.S_hh.w_inh = W_HH_INIT[h.src_hh, h.tgt_hh]
    dummy = TimedArray(np.zeros((5, N_IN), dtype=float), dt=DT_SIM)
    h.G_in.namespace["I_timed"] = dummy
    h.net.run(5 * ms)
    h.net.restore('init')


# ═══════════════════════════════════════════════════════════════════════════════
# Per-wav training
# ═══════════════════════════════════════════════════════════════════════════════

def train_fingerprint(h, wav_path):
    """Run N_EPOCHS worth of the (CLIP_MS-truncated) wav as ONE continuous net.run()
    and return (w_ih_final, w_hh_final, in_counts_final, hid_counts_final), or None.

    w_ih_final, w_hh_final : final synaptic weights after the whole run. State is
        never reset between epochs, so this already IS the last epoch's result —
        there is nothing to snapshot or average.
    in_counts_final, hid_counts_final : (N_IN,)/(N_H,) spike counts from the LAST
        epoch's time window only, [(N_EPOCHS-1)*T, N_EPOCHS*T), isolated by slicing
        the recorded spike times after the run completes.
    """
    try:
        I, T = compute_spike_input_current(wav_path, **ENCODER_KWARGS)
    except Exception as e:
        print(f"  [skip {wav_path}: {e}]")
        return None

    if T > CLIP_MS:
        T = CLIP_MS
        I = I[:, :T]

    h.net.restore('init')
    I_tiled = np.tile(I, (1, N_EPOCHS))          # (N_IN, T * N_EPOCHS)
    h.G_in.namespace["I_timed"] = TimedArray(I_tiled.T.astype(float), dt=DT_SIM)

    h.S_ih.w     = W_IH_INIT[h.src_ih, h.tgt_ih]
    h.S_ih.apre  = 0
    h.S_ih.apost = 0
    h.S_ih.r1 = 0; h.S_ih.r2 = 0; h.S_ih.o1 = 0; h.S_ih.o2 = 0   # triplet detectors
    h.S_hh.w_inh     = W_HH_INIT[h.src_hh, h.tgt_hh]
    h.S_hh.apre_inh  = 0
    h.S_hh.apost_inh = 0

    # ONE continuous run over every epoch — no restore, no reassignment, no rebuild
    # between epochs. N_EPOCHS separate net.run() calls (each with Python-side
    # overhead) collapse into a single call over the whole tiled input.
    h.net.run(N_EPOCHS * T * DT_SIM)

    w_ih_final = np.zeros((N_IN, N_H), dtype=np.float32)
    w_ih_final[h.src_ih, h.tgt_ih] = np.array(h.S_ih.w)
    w_hh_final = np.zeros((N_H, N_H), dtype=np.float32)
    w_hh_final[h.src_hh, h.tgt_hh] = np.array(h.S_hh.w_inh)

    final_epoch_start_ms = (N_EPOCHS - 1) * T
    in_t  = np.array(h.spike_in.t / ms)
    hid_t = np.array(h.spike_hid.t / ms)
    in_i  = np.array(h.spike_in.i)
    hid_i = np.array(h.spike_hid.i)
    in_counts_final  = np.bincount(in_i[in_t >= final_epoch_start_ms],
                                    minlength=N_IN).astype(np.int64)
    hid_counts_final = np.bincount(hid_i[hid_t >= final_epoch_start_ms],
                                    minlength=N_H).astype(np.int64)

    return w_ih_final, w_hh_final, in_counts_final, hid_counts_final


# ═══════════════════════════════════════════════════════════════════════════════
# Tensor transform → ANN-ready per-sample output
# ═══════════════════════════════════════════════════════════════════════════════

def _normalize_activity(counts):
    """Per-sample [0,1] via ÷99th-percentile + clip. Preserves 0 (silent)."""
    denom = np.percentile(counts, 99)
    if denom <= 0:
        return np.zeros_like(counts, dtype=np.float16)
    return np.clip(counts / denom, 0.0, 1.0).astype(np.float16)


def fingerprint_to_sample(w_ih, w_hh, in_counts, hid_counts):
    """(N_IN,N_H) + (N_H,N_H) final weight matrices + final-epoch spike counts ->
    (in_weights, hid_weights, input_activity, hidden_activity).

    in_weights  (2,23,128) float16 — in->hid type-images, silent cells zeroed
    hid_weights (2,23,128) float16 — hid->hid type-images, silent cells zeroed
                                      (self-connections are already 0 by construction)
    input_activity  (128,) float16 — final-epoch [0,1]
    hidden_activity (128,) float16 — final-epoch [0,1]
    """
    in_silent  = (in_counts == 0)
    hid_silent = (hid_counts == 0)

    # Zero any weight whose presynaptic input neuron OR postsynaptic hidden neuron
    # never fired in the final epoch.
    in_weights = w_ih[IN_IDX, _J_ROW].astype(np.float32)            # (2,23,N_H)
    in_weights[in_silent[IN_IDX] | hid_silent[None, None, :]] = 0.0

    # Same, but both endpoints are hidden neurons.
    hid_weights = w_hh[HH_IDX, _J_ROW].astype(np.float32)           # (2,23,N_H)
    hid_weights[hid_silent[HH_IDX] | hid_silent[None, None, :]] = 0.0

    return (in_weights.astype(np.float16),
            hid_weights.astype(np.float16),
            _normalize_activity(in_counts),
            _normalize_activity(hid_counts))


# ═══════════════════════════════════════════════════════════════════════════════
# Multiprocessing worker helpers (imported by spawn workers from this real module)
# ═══════════════════════════════════════════════════════════════════════════════

_H = None   # per-process Brian2 network handle


def _init_worker():
    """Pool initializer: build one Brian2 network per worker process and warm its
    Cython codegen cache before any real wav is processed."""
    global _H
    _H = build_network()
    warmup(_H)


def process_one(entry):
    """Generate one fingerprint. `entry` is a plain dict (picklable for spawn).

    Always returns (person_id, session_id, file_name, label, payload) so the driver
    can track per-person completion even for skipped wavs. `payload` is
    (in_weights, hid_weights, input_activity, hidden_activity) on success, or None if
    the wav could not be encoded.
    """
    pid, sid, fname, lab = (entry['person_id'], entry['session_id'],
                             entry['file_name'], entry['label'])
    out = train_fingerprint(_H, entry['wav_path'])
    if out is None:
        return (pid, sid, fname, lab, None)
    iw, hw, ia, ha = fingerprint_to_sample(*out)
    return (pid, sid, fname, lab, (iw, hw, ia, ha))


In [ ]:
# ── Enumerate wavs: every session, ONE random wav each (seeded) ────────────────
# Matches the corpus-wide "1 random utterance per session" protocol used by every
# other pipeline in this project (see ecapa_baseline.ipynb's enumerate_corpus).
# Picking randomly (not first-sorted) means the SNN sees the same distribution of
# utterances as ECAPA training does — file_name is recorded per entry below so
# downstream code (the FiLM-conditioned ECAPA notebook) can look up the exact
# audio each fingerprint came from without re-deriving this pick itself.
from pathlib import Path
import numpy as np

root = Path(INPUT_ROOT)
assert root.is_dir(), f"INPUT_ROOT not found: {root}"

rng = np.random.default_rng(SEED)
entries = []
for person_dir in sorted(d for d in root.iterdir() if d.is_dir()):
    for sess_dir in sorted(d for d in person_dir.iterdir() if d.is_dir()):
        wavs = sorted(f.name for f in sess_dir.iterdir()
                      if f.is_file() and f.suffix.lower() in AUDIO_EXTS)
        if not wavs:
            continue
        wav = wavs[rng.integers(len(wavs))]
        entries.append(dict(
            person_id=person_dir.name,
            session_id=sess_dir.name,
            file_name=wav,
            label=f"{person_dir.name}/{sess_dir.name}/{wav}",
            wav_path=str(sess_dir / wav),
        ))

n_persons  = len({e["person_id"] for e in entries})
n_sessions = len({(e["person_id"], e["session_id"]) for e in entries})
print(f"{len(entries)} wavs  |  {n_persons} persons  |  {n_sessions} sessions "
      f"(1 random utterance per session)")
assert entries, "no audio found under INPUT_ROOT — check the path / AUDIO_EXTS"

In [ ]:
# ── Generate fingerprints in parallel (spawn), then write ONE npz ─────────────
# Logs one line each time a person's wavs are all done (successful + skipped).
import multiprocessing as mp
import time
from collections import defaultdict
import numpy as np
import _fingerprint_core as C

# Expected wavs per person → lets us detect when a person is fully processed even
# though imap_unordered returns results in arbitrary order.
expected = defaultdict(int)
for e in entries:
    expected[e["person_id"]] += 1
n_persons_total = len(expected)

t0 = time.time()
print("warming Brian2 codegen cache (parent process) ...", flush=True)
_h_warm = C.build_network()
C.warmup(_h_warm)                       # compiles once in the parent; workers reuse the cache
del _h_warm
print(f"  warm-up done in {time.time()-t0:.0f}s")

results = []                            # (iw, hw, ia, ha, pid, sid, fname, label)
processed = defaultdict(int)
persons_done = n_wavs_done = n_skipped = 0

ctx = mp.get_context("spawn")
with ctx.Pool(WORKERS, initializer=C._init_worker) as pool:
    for pid, sid, fname, lab, payload in pool.imap_unordered(C.process_one, entries):
        n_wavs_done += 1
        processed[pid] += 1
        if payload is not None:         # None = unencodable wav (skipped)
            iw, hw, ia, ha = payload
            results.append((iw, hw, ia, ha, pid, sid, fname, lab))
        else:
            n_skipped += 1
        if processed[pid] == expected[pid]:
            persons_done += 1
            print(f"[person {persons_done}/{n_persons_total}] {pid} done  |  "
                  f"{len(results)} fingerprints, {n_skipped} skipped, "
                  f"{n_wavs_done}/{len(entries)} wavs  |  {time.time()-t0:.0f}s",
                  flush=True)

assert results, "no fingerprints produced"
print(f"produced {len(results)} fingerprints ({n_skipped} skipped) "
      f"over {n_persons_total} persons in {time.time()-t0:.0f}s")

# ── Stack + save ────────────────────────────────────────────────────────────────
in_weights      = np.stack([r[0] for r in results]).astype(np.float16)   # (N,2,23,128)
hid_weights     = np.stack([r[1] for r in results]).astype(np.float16)   # (N,2,23,128)
input_activity  = np.stack([r[2] for r in results]).astype(np.float16)   # (N,128)
hidden_activity = np.stack([r[3] for r in results]).astype(np.float16)   # (N,128)
person_ids  = np.array([r[4] for r in results])
session_ids = np.array([r[5] for r in results])
file_names  = np.array([r[6] for r in results])
labels      = np.array([r[7] for r in results])

os.makedirs(WORK_DIR, exist_ok=True)
out_path = os.path.join(WORK_DIR, OUTPUT_NAME)
np.savez(out_path,
         in_weights=in_weights, hid_weights=hid_weights,
         input_activity=input_activity, hidden_activity=hidden_activity,
         person_ids=person_ids, session_ids=session_ids,
         file_names=file_names, labels=labels)
print(f"saved → {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB)")

In [ ]:
# ── Verify the written npz ────────────────────────────────────────────────────
import numpy as np
d = np.load(out_path, allow_pickle=True)
print("keys:", list(d.keys()))
for k in ("in_weights", "hid_weights", "input_activity", "hidden_activity"):
    a = d[k]
    print(f"  {k:16s} {str(a.shape):20s} {a.dtype}  "
          f"range [{float(a.min()):.4f}, {float(a.max()):.4f}]")
print("  samples :", len(d["labels"]))
print("  persons :", len(set(d["person_ids"].tolist())))
print("  sessions:", len({(p, s) for p, s in zip(d["person_ids"], d["session_ids"])}))
print("  files   :", d["file_names"][:5].tolist(), "...")
assert 0.0 <= float(d["input_activity"].min()) and float(d["input_activity"].max()) <= 1.0
assert 0.0 <= float(d["hidden_activity"].min()) and float(d["hidden_activity"].max()) <= 1.0
print("OK — activities in [0,1]; ready to download and bundle into a Kaggle dataset.")